In [0]:
import numpy as np
import pandas as pd # Data processing, CSV file I/O (e.g. pd.read_csv)
df=pd.read_csv('/Workspace/Users/silvesterkisalu@gmail.com/owid-energy-data.csv') # Replace the file path here with the workspace path you copied earlier

In [0]:
display(df)

Databricks data profile. Run in Databricks to view.

In [0]:
df.shape

In [0]:
df.describe()

In [0]:
df.dtypes

In [0]:
# Missing values count and percentage
missing_data = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2)
})

# Sort by missing count and show only columns with missing values
missing_data = missing_data[missing_data['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)
print(f"Columns with missing values: {len(missing_data)} out of {len(df.columns)}")
display(missing_data.head(20))

In [0]:
# Data types distribution
print("Data Types Summary:")
print(df.dtypes.value_counts())
print(f"\nTotal columns: {len(df.columns)}")
print(f"\nColumns by type:")
for dtype in df.dtypes.value_counts().index:
    cols = df.select_dtypes(include=[dtype]).columns.tolist()
    print(f"\n{dtype}: {len(cols)} columns")
    print(f"  {', '.join(cols[:5])}{'...' if len(cols) > 5 else ''}")

In [0]:
# Unique values count for each column
unique_counts = pd.DataFrame({
    'Column': df.columns,
    'Unique_Values': [df[col].nunique() for col in df.columns],
    'Unique_Percentage': [round(df[col].nunique() / len(df) * 100, 2) for col in df.columns]
})

unique_counts = unique_counts.sort_values('Unique_Values', ascending=False)
print(f"Total rows: {len(df)}")
print(f"\nColumns with high cardinality (>50% unique):")
high_card = unique_counts[unique_counts['Unique_Percentage'] > 50].head(10)
if len(high_card) > 0:
    display(high_card)
else:
    print("  None found")
    
print(f"\nColumns with low cardinality (<10 unique values):")
low_card = unique_counts[unique_counts['Unique_Values'] < 10]
if len(low_card) > 0:
    display(low_card)
else:
    print("  None found")

In [0]:
# Extended statistics for numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns
print(f"Analyzing {len(numeric_cols)} numeric columns\n")

# Get describe with additional percentiles
stats = df[numeric_cols].describe(percentiles=[.05, .25, .5, .75, .95]).T
stats['missing'] = df[numeric_cols].isnull().sum()
stats['missing_pct'] = (stats['missing'] / len(df) * 100).round(2)

display(stats[['count', 'missing', 'missing_pct', 'mean', 'std', 'min', '5%', '50%', '95%', 'max']].head(20))

In [0]:
# Check for duplicate columns
duplicate_columns = df.columns[df.columns.duplicated()].tolist()
if duplicate_columns:
    print(f"Duplicate columns found: {duplicate_columns}")
    df = df.loc[:, ~df.columns.duplicated()]
else:
    print("No duplicate columns found.")

# Check for duplicate rows
duplicate_rows = df[df.duplicated()]
if not duplicate_rows.empty:
    print(f"Duplicate rows found: {len(duplicate_rows)}")
    display(duplicate_rows)
    df = df.drop_duplicates()
else:
    print("No duplicate rows found.")

In [0]:
# Check for date columns
# This dataset has no date columns - only a 'year' integer column (1900-2023)
# If you need a date column, you can create one:

# Option 1: Create a date column from the year (January 1st of each year)
df['date'] = pd.to_datetime(df['year'], format='%Y')

# Option 2: If you want a specific date format as string
# df['date_str'] = df['year'].astype(str) + '-01-01'

print(f"Created 'date' column:")
print(df[['country', 'year', 'date']].head())
print(f"\nDate column type: {df['date'].dtype}")

In [0]:
# Drop the column you don't need (replace 'XXX' with actual column name)
# Example: df = df.drop(columns=['iso_code'])

# Uncomment and modify the line below with the actual column name:
# df = df.drop(columns=['XXX'])

print("Specify which column to drop by uncommenting and editing the line above.")
print(f"\nCurrent columns: {list(df.columns)}")

In [0]:
# Check for duplicate columns
duplicate_columns = df.columns[df.columns.duplicated()].tolist()
if duplicate_columns:
    print(f"Duplicate columns found: {duplicate_columns}")
    df = df.loc[:, ~df.columns.duplicated()]
    print(f"Removed duplicate columns. Current shape: {df.shape}")
else:
    print("No duplicate columns found.")

# Check for duplicate rows
duplicate_rows = df[df.duplicated()]
if not duplicate_rows.empty:
    print(f"\nDuplicate rows found: {len(duplicate_rows)}")
    df = df.drop_duplicates()
    print(f"Removed duplicate rows. Current shape: {df.shape}")
else:
    print("\nNo duplicate rows found.")

print(f"\nFinal DataFrame shape: {df.shape}")

In [0]:
df = df.fillna(0) # Replace all NaN (Not a Number) values with 0

In [0]:
# Ensure the 'year' column is converted to the correct data type (integer for year)
df['year'] = pd.to_datetime(df['year'], format='%Y', errors='coerce').dt.year

# Confirm the changes
df.year.dtype

In [0]:
# Get the unique countries
unique_countries = df['country'].unique()
unique_countries

In [0]:
import plotly.express as px

# Filter data to include only years from 2000 to 2022
filtered_data = df[(df['year'] >= 2000) & (df['year'] <= 2022)]

# Get the top 10 countries with the highest emissions in the filtered data
top_countries = filtered_data.groupby('country')['greenhouse_gas_emissions'].sum().nlargest(10).index

# Filter the data for those top countries
top_countries_data = filtered_data[filtered_data['country'].isin(top_countries)]

# Plot emissions trends over time for these countries
fig = px.line(top_countries_data, x='year', y='greenhouse_gas_emissions', color='country',
             title="Greenhouse Gas Emissions Trends for Top 10 Countries (2000 - 2022)")
fig.show()

In [0]:
# Filter out regional entities
regions = ['Africa', 'Asia', 'Europe', 'North America', 'South America', 'Oceania']

# Calculate total emissions for each region
regional_emissions = df[df['country'].isin(regions)].groupby('country')['greenhouse_gas_emissions'].sum()

# Plot the comparison
fig = px.bar(regional_emissions, title="Greenhouse Gas Emissions by Region")
fig.show()

In [0]:
# Calculate the renewable energy share and save it as a new column called "renewable_share"
df['renewable_share'] = df['renewables_consumption'] / df['primary_energy_consumption']

# Rank countries by their average renewable energy share
renewable_ranking = df.groupby('country')['renewable_share'].mean().sort_values(ascending=False)

# Filter for countries leading in renewable energy share
leading_renewable_countries = renewable_ranking.head(10).index
leading_renewable_data = df[df['country'].isin(leading_renewable_countries)]
# filtered_data = df[(df['year'] >= 2000) & (df['year'] <= 2022)]
leading_renewable_data_filter=leading_renewable_data[(leading_renewable_data['year'] >= 2000) & (leading_renewable_data['year'] <= 2022)]
# Plot renewable share over time for top renewable countries
fig = px.line(leading_renewable_data_filter, x='year', y='renewable_share', color='country',
             title="Renewable Energy Share Growth Over Time for Leading Countries")
fig.show()

In [0]:
# Select top emitters and calculate renewable share vs. emissions
top_emitters = df.groupby('country')['greenhouse_gas_emissions'].sum().nlargest(10).index
top_emitters_data = df[df['country'].isin(top_emitters)]

# Plot renewable share vs. greenhouse gas emissions over time
fig = px.scatter(top_emitters_data, x='renewable_share', y='greenhouse_gas_emissions',
                color='country', title="Impact of Renewable Energy on Emissions for Top Emitters")
fig.show()

In [0]:
%pip install statsmodels

import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
import matplotlib.pyplot as plt

# Aggregate global primary energy consumption by year
global_energy = df[df['country'] == 'World'].groupby('year')['primary_energy_consumption'].sum()

# Build an ARIMA model for projection
model = ARIMA(global_energy, order=(1, 1, 1))
model_fit = model.fit()
forecast = model_fit.forecast(steps=10)  # Projecting for 10 years

# Plot historical and forecasted energy consumption
plt.plot(global_energy, label='Historical')
plt.plot(range(global_energy.index[-1] + 1, global_energy.index[-1] + 11), forecast, label='Forecast')
plt.xlabel("Year")
plt.ylabel("Primary Energy Consumption")
plt.title("Projected Global Energy Consumption")
plt.legend()
plt.show()

In [0]:
%pip install kaleido

In [0]:
import os
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

# Create visualizations directory in the Git repo
output_dir = '/Workspace/Repos/silvesterkisalu@gmail.com/BigDataAnalytics/visualizations'
os.makedirs(output_dir, exist_ok=True)

print("Generating visualization images...\n")

# 1. Greenhouse Gas Emissions Trends for Top 10 Countries (2000-2022)
filtered_data = df[(df['year'] >= 2000) & (df['year'] <= 2022)]
top_countries = filtered_data.groupby('country')['greenhouse_gas_emissions'].sum().nlargest(10).index
top_countries_data = filtered_data[filtered_data['country'].isin(top_countries)]

plt.figure(figsize=(14, 8))
for country in top_countries:
    country_data = top_countries_data[top_countries_data['country'] == country]
    plt.plot(country_data['year'], country_data['greenhouse_gas_emissions'], 
             marker='o', label=country, linewidth=2, markersize=4)
plt.xlabel('Year', fontsize=13)
plt.ylabel('Greenhouse Gas Emissions', fontsize=13)
plt.title('Greenhouse Gas Emissions Trends for Top 10 Countries (2000-2022)', 
          fontsize=15, fontweight='bold', pad=20)
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{output_dir}/01_ghg_emissions_top10_countries.png", dpi=150, bbox_inches='tight')
plt.close()
print("✓ Saved: 01_ghg_emissions_top10_countries.png")

# 2. Greenhouse Gas Emissions by Region
regions = ['Africa', 'Asia', 'Europe', 'North America', 'South America', 'Oceania']
regional_emissions = df[df['country'].isin(regions)].groupby('country')['greenhouse_gas_emissions'].sum().sort_values(ascending=True)

plt.figure(figsize=(12, 7))
colors = cm.viridis(np.linspace(0.2, 0.9, len(regional_emissions)))
bars = plt.barh(regional_emissions.index, regional_emissions.values, color=colors)
plt.xlabel('Total Greenhouse Gas Emissions', fontsize=13)
plt.ylabel('Region', fontsize=13)
plt.title('Greenhouse Gas Emissions by Region', fontsize=15, fontweight='bold', pad=20)
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(f"{output_dir}/02_ghg_emissions_by_region.png", dpi=150, bbox_inches='tight')
plt.close()
print("✓ Saved: 02_ghg_emissions_by_region.png")

# 3. Renewable Energy Share Growth Over Time
renewable_ranking = df.groupby('country')['renewable_share'].mean().sort_values(ascending=False)
leading_renewable_countries = renewable_ranking.head(10).index
leading_renewable_data = df[df['country'].isin(leading_renewable_countries)]
leading_renewable_data_filter = leading_renewable_data[(leading_renewable_data['year'] >= 2000) & (leading_renewable_data['year'] <= 2022)]

plt.figure(figsize=(14, 8))
for country in leading_renewable_countries:
    country_data = leading_renewable_data_filter[leading_renewable_data_filter['country'] == country]
    if not country_data.empty:
        plt.plot(country_data['year'], country_data['renewable_share'], 
                marker='o', label=country, linewidth=2, markersize=4)
plt.xlabel('Year', fontsize=13)
plt.ylabel('Renewable Energy Share', fontsize=13)
plt.title('Renewable Energy Share Growth Over Time for Leading Countries', 
          fontsize=15, fontweight='bold', pad=20)
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{output_dir}/03_renewable_share_growth.png", dpi=150, bbox_inches='tight')
plt.close()
print("✓ Saved: 03_renewable_share_growth.png")

# 4. Impact of Renewable Energy on Emissions
top_emitters = df.groupby('country')['greenhouse_gas_emissions'].sum().nlargest(10).index
top_emitters_data = df[df['country'].isin(top_emitters)]

plt.figure(figsize=(14, 8))
for i, country in enumerate(top_emitters):
    country_data = top_emitters_data[top_emitters_data['country'] == country]
    plt.scatter(country_data['renewable_share'], country_data['greenhouse_gas_emissions'], 
               label=country, s=50, alpha=0.6)
plt.xlabel('Renewable Energy Share', fontsize=13)
plt.ylabel('Greenhouse Gas Emissions', fontsize=13)
plt.title('Impact of Renewable Energy on Emissions for Top Emitters', 
          fontsize=15, fontweight='bold', pad=20)
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{output_dir}/04_renewable_vs_emissions.png", dpi=150, bbox_inches='tight')
plt.close()
print("✓ Saved: 04_renewable_vs_emissions.png")

# 5. ARIMA Forecast - Projected Global Energy Consumption
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA

global_energy = df[df['country'] == 'World'].groupby('year')['primary_energy_consumption'].sum()
model = ARIMA(global_energy, order=(1, 1, 1))
model_fit = model.fit()
forecast = model_fit.forecast(steps=10)

plt.figure(figsize=(12, 7))
plt.plot(global_energy, label='Historical', linewidth=2)
plt.plot(range(global_energy.index[-1] + 1, global_energy.index[-1] + 11), forecast, 
         label='Forecast', linewidth=2, linestyle='--')
plt.xlabel("Year", fontsize=12)
plt.ylabel("Primary Energy Consumption", fontsize=12)
plt.title("Projected Global Energy Consumption", fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{output_dir}/05_arima_forecast_global_energy.png", dpi=150, bbox_inches='tight')
plt.close()
print("✓ Saved: 05_arima_forecast_global_energy.png")

print(f"\n✓ All 5 visualizations exported to: {output_dir}")
print("\nFiles created:")
for file in sorted(os.listdir(output_dir)):
    print(f"  - {file}")